<div style="background:#123B63;color:white;padding:14px 18px;border-radius:6px">
<b>CSE 816 &mdash; Machine Learning Lab</b> &nbsp;&middot;&nbsp; Department of CSE, University of Chittagong<br>
<span style="font-size:90%">Module 1 &middot; Week 1 &middot; Part 3 of 4 &nbsp;&middot;&nbsp; 60 minutes</span>
</div>

# Cost Function Implementation

In Part 2 you judged a model by eye. That does not scale and it does not settle arguments.
Here you build the number that decides which of two models is better, and then look at the
shape of that number as the parameters vary &mdash; the landscape gradient descent will walk.

**Companion theory lecture:** CSE 815, Week 1, Session 1 (the cost function) and Session 2 (visualising it).

## What you will be able to do

1. Implement $J(w,b)$ from the formula, with a loop and vectorised.
2. Verify your implementation against a case whose answer you can work out by hand.
3. Plot $J(w)$ for a fixed $b$ and recognise the parabola.
4. Draw $J(w,b)$ as a surface and as a contour map, and read a model's quality off it.

---

## 1. The formula

$$J(w,b) \;=\; \frac{1}{2m}\sum_{i=1}^{m}\bigl(f_{w,b}(x^{(i)}) - y^{(i)}\bigr)^{2}$$

Reading it right to left, in the order you will code it:

1. $f_{w,b}(x^{(i)}) - y^{(i)}$ &mdash; the residual for one example.
2. $(\cdot)^2$ &mdash; square it, so sign does not matter and big misses hurt more.
3. $\sum_{i=1}^{m}$ &mdash; add up over all examples.
4. $\frac{1}{2m}$ &mdash; average (the $m$), and halve (the $2$, which will cancel against the
   exponent when we differentiate in Part 4).

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from mpl_toolkits.mplot3d import Axes3D          # noqa: F401  (registers the 3-D projection)

%matplotlib inline
try:
    plt.style.use("seaborn-v0_8-whitegrid")
except OSError:
    plt.style.use("ggplot")

x_train = np.array([0.8, 1.0, 1.2, 1.4, 1.6, 1.8, 2.0, 2.4])
y_train = np.array([30., 36., 45., 48., 58., 62., 70., 82.])
m = x_train.shape[0]

print("m =", m)

### Version 1: with a loop, following the formula term by term

In [ ]:
def compute_cost_loop(x, y, w, b):
    """
    Squared-error cost of the model f(x) = wx + b, computed with a loop.

    Args:
        x (ndarray (m,)): input features
        y (ndarray (m,)): true targets
        w, b (scalar)   : model parameters

    Returns:
        total_cost (float): the value of J(w, b)
    """
    m = x.shape[0]
    cost_sum = 0.0
    for i in range(m):
        f_wb_i = w * x[i] + b            # prediction for example i
        cost_sum += (f_wb_i - y[i]) ** 2  # squared residual
    return cost_sum / (2 * m)


print("J(30, 5) =", compute_cost_loop(x_train, y_train, 30.0, 5.0))

### Version 2: vectorised

In [ ]:
def compute_cost(x, y, w, b):
    """
    Squared-error cost of the model f(x) = wx + b (vectorised).

    Args:
        x (ndarray (m,)): input features
        y (ndarray (m,)): true targets
        w, b (scalar)   : model parameters

    Returns:
        total_cost (float): the value of J(w, b)
    """
    m = x.shape[0]
    f_wb = w * x + b                      # all m predictions at once
    return np.sum((f_wb - y) ** 2) / (2 * m)


print("J(30, 5) =", compute_cost(x_train, y_train, 30.0, 5.0))
assert np.isclose(compute_cost(x_train, y_train, 30.0, 5.0),
                  compute_cost_loop(x_train, y_train, 30.0, 5.0))
print("the two implementations agree")

---

## 2. Test it against an answer you already know

This is the habit that separates working code from code that merely runs. Pick a case
whose answer you can derive on paper, then assert it.

From the lecture: for the data $(1,1), (2,2), (3,3)$ and the model $f_w(x) = wx$ (so $b=0$),

$$J(w) = \tfrac{1}{6}\bigl[(w-1)^2 + (2w-2)^2 + (3w-3)^2\bigr] = \tfrac{14}{6}(w-1)^2 .$$

That gives three checkable values: $J(1) = 0$, $J(0.5) \approx 0.5833$, $J(0) \approx 2.3333$.

In [ ]:
x_toy = np.array([1.0, 2.0, 3.0])
y_toy = np.array([1.0, 2.0, 3.0])

for w_test, expected in [(1.0, 0.0), (0.5, 14 / 6 * 0.25), (0.0, 14 / 6)]:
    got = compute_cost(x_toy, y_toy, w_test, 0.0)
    ok = np.isclose(got, expected)
    print(f"J({w_test}) = {got:.4f}   expected {expected:.4f}   {'PASS' if ok else 'FAIL'}")
    assert ok

print("\nAll analytic checks passed.")

Three properties worth noticing, and worth stating in a viva:

- $J \geq 0$ always, because it is a sum of squares.
- $J = 0$ exactly when the line passes through every point &mdash; possible here, rare in
  real data.
- $J$ is measured in the *square* of the target's units. For our flats that is
  $(\text{lakh BDT})^2$, which is why the raw value of $J$ is not directly interpretable.
  Only *comparisons* between values of $J$ matter.

---

## 3. Comparing models with a number

Back to the flats. In Part 2 you compared three lines by eye. Now compare them by cost.

In [ ]:
candidates = [(20.0, 5.0), (30.0, 5.0), (33.0, 4.0), (45.0, 5.0)]

print(f"{'w':>6} {'b':>6} {'J(w,b)':>12}")
print("-" * 26)
for w_c, b_c in candidates:
    print(f"{w_c:6.1f} {b_c:6.1f} {compute_cost(x_train, y_train, w_c, b_c):12.4f}")

The ranking now needs no argument: lower $J$ wins.

---

## 4. The shape of $J$

### One parameter at a time

Freeze $b$ and sweep $w$. The lecture said this traces a parabola; confirm it.

In [ ]:
b_fixed = 4.0
w_values = np.linspace(0, 70, 200)
J_values = np.array([compute_cost(x_train, y_train, w_i, b_fixed) for w_i in w_values])

w_best = w_values[np.argmin(J_values)]

plt.figure(figsize=(6.5, 4))
plt.plot(w_values, J_values, c="#123B63", lw=2)
plt.scatter([w_best], [J_values.min()], c="#9B2C4B", s=70, zorder=5,
            label=f"minimum near w = {w_best:.1f}")
plt.title(f"J(w) with b fixed at {b_fixed}")
plt.xlabel("w")
plt.ylabel("J(w, b=4)")
plt.legend()
plt.show()

print(f"lowest cost on this grid: J({w_best:.2f}, {b_fixed}) = {J_values.min():.4f}")

A parabola, opening upwards, with exactly one lowest point. That is what **convex** means
here, and it is the reason gradient descent cannot get stuck in Part 4.

Note the sweep found `w` on a grid of 200 values. That is fine for one parameter. With two
parameters it costs $200^2$ evaluations; with a neural network's million parameters it is
impossible. Grid search is a teaching device, not a training algorithm.

### Both parameters: the bowl

Now let both $w$ and $b$ vary. `np.meshgrid` builds the grid of parameter pairs.

In [ ]:
w_grid = np.linspace(10, 60, 120)
b_grid = np.linspace(-25, 35, 120)
W, B = np.meshgrid(w_grid, b_grid)

# Evaluate J at every (w, b) pair on the grid.
J_grid = np.zeros_like(W)
for r in range(W.shape[0]):
    for c in range(W.shape[1]):
        J_grid[r, c] = compute_cost(x_train, y_train, W[r, c], B[r, c])

# Locate the best grid point.
r_min, c_min = np.unravel_index(np.argmin(J_grid), J_grid.shape)
w_star, b_star = W[r_min, c_min], B[r_min, c_min]
print(f"best on grid: w = {w_star:.2f}, b = {b_star:.2f}, J = {J_grid.min():.4f}")

In [ ]:
fig = plt.figure(figsize=(12, 4.5))

# --- the surface ---
ax1 = fig.add_subplot(1, 2, 1, projection="3d")
ax1.plot_surface(W, B, J_grid, cmap="viridis", alpha=0.9, linewidth=0)
ax1.set_xlabel("w")
ax1.set_ylabel("b")
ax1.set_zlabel("J(w, b)")
ax1.set_title("J as a surface")
ax1.view_init(elev=30, azim=-125)

# --- the contours ---
ax2 = fig.add_subplot(1, 2, 2)
levels = np.logspace(np.log10(max(J_grid.min(), 1e-2)), np.log10(J_grid.max()), 18)
cs = ax2.contour(W, B, J_grid, levels=levels, cmap="viridis")
ax2.scatter([w_star], [b_star], c="#9B2C4B", s=80, marker="*", zorder=5,
            label="grid minimum")
ax2.set_xlabel("w")
ax2.set_ylabel("b")
ax2.set_title("The same J, seen from above")
ax2.legend()

plt.tight_layout()
plt.show()

Two things to take from the contour plot:

- Each closed curve joins parameter pairs of **equal cost** &mdash; contour lines on a map.
  Moving to an inner contour means a better model.
- The contours are long thin **ellipses**, not circles, and they are tilted. That says $w$
  and $b$ are correlated: raising the slope can be partly compensated by lowering the
  intercept. It also means the steepest-descent direction usually does *not* point straight
  at the minimum, which is why gradient descent will zig-zag in Part 4. Feature scaling in
  Week 2 makes these ellipses rounder.

### Where your candidates sit

In [ ]:
plt.figure(figsize=(6.5, 5))
plt.contour(W, B, J_grid, levels=levels, cmap="viridis", alpha=0.7)

for w_c, b_c in candidates:
    J_c = compute_cost(x_train, y_train, w_c, b_c)
    plt.scatter([w_c], [b_c], s=70, zorder=5)
    plt.annotate(f"({w_c:g}, {b_c:g})\nJ={J_c:.1f}", (w_c, b_c),
                 textcoords="offset points", xytext=(8, 6), fontsize=8)

plt.scatter([w_star], [b_star], c="#9B2C4B", s=120, marker="*", zorder=6,
            label="grid minimum")
plt.xlabel("w")
plt.ylabel("b")
plt.title("Candidate models in parameter space")
plt.legend()
plt.show()

### Checkpoint 1

Write a function `rank_models(x, y, params)` that takes a list of `(w, b)` tuples and prints
them sorted from lowest to highest cost. Run it on `candidates` plus the best `(w, b)` you
found by hand in Part 2.

Did your eye pick the same winner the cost function does?

In [ ]:
# Your code here

### Checkpoint 2

The $\frac{1}{2}$ in the formula is a convenience, not a requirement. Write
`compute_cost_no_half(x, y, w, b)` without it, evaluate both versions over `w_values` with
`b_fixed`, and confirm that **the minimising $w$ is the same** even though every cost value
doubles.

This is worth doing once: it shows that scaling a cost function by a positive constant
changes the number but not the answer.

In [ ]:
# Your code here

---

## 5. Recap

- $J(w,b)$ is the mean squared residual, halved: `np.sum((w*x + b - y)**2) / (2*m)`.
- Test new numerical code against a hand-derived value before trusting it.
- With $b$ fixed, $J(w)$ is a parabola; with both free, $J(w,b)$ is a convex bowl.
- Contours are equal-cost curves; inner means better. Tilted ellipses mean correlated
  parameters.
- Grid search finds the minimum here but is useless at scale &mdash; hence Part 4.

### Exercises to hand in

1. Add an outlier: a 1.2k sq ft flat priced at 150 lakh. Recompute the cost of
   `(33, 4)` and re-run the grid search. How far does the minimum move? Explain the size of
   the shift in terms of the **squaring** in $J$.
2. Implement `compute_cost_mae` using $\frac{1}{m}\sum|f - y|$ instead. Plot it against $w$
   beside the squared-error version, on the outlier-contaminated data from exercise 1. Which
   is less disturbed by the outlier, and why?
3. From the lecture's design question: a delivery-time model where being 10 minutes late is
   worse than being 10 minutes early. Implement an asymmetric cost that charges late
   predictions three times as much, plot it against $w$, and state whether your version is
   still (a) non-negative, (b) differentiable everywhere, (c) convex.
4. Explain in a short paragraph why $J$ has units of $(\text{lakh BDT})^2$, and what that
   implies about reporting "the cost is 12.4" to a non-technical stakeholder.

### Next

**Part 4 &mdash; Gradient Descent:** stop searching the grid and start walking downhill.

**Reading:** James et al., *ISL* 2e, &sect;3.1.1; Bishop, *PRML*, &sect;1.1.